# Derivatives, gradients, and stochastic gradient descent

Why a differentiable loss makes learning tractable, and what backpropagation is doing when Keras calls fit().

**Runs on:** CPU — about 1 minute &nbsp;·&nbsp; **Slides:** [Chapter 2 — The Mathematical Building Blocks of Neural Networks](../../../course-web-slides/ch02/index.html) &nbsp;·&nbsp; **Section:** 03 — The engine of neural networks

---

## A derivative, numerically and then exactly

In [ ]:
import numpy as np

f = lambda x: x ** 2 + 3 * x + 1
df_exact = lambda x: 2 * x + 3

x0, eps = 2.0, 1e-6
numeric = (f(x0 + eps) - f(x0)) / eps
print(f"numeric: {numeric:.6f}   exact: {df_exact(x0):.6f}")

The derivative says: *move x a little, and f moves this much, in this direction*. Learning is nothing more than using that to decide which way to move.

## Gradient descent by hand

In [ ]:
import matplotlib.pyplot as plt

x = 4.0
lr = 0.15
path = [x]
for _ in range(25):
    x = x - lr * df_exact(x)
    path.append(x)

xs = np.linspace(-6, 5, 300)
plt.figure(figsize=(6, 4))
plt.plot(xs, f(xs), lw=1.5)
plt.plot(path, [f(p) for p in path], "o-", ms=4, lw=.8, color="#c0392b")
plt.title(f"25 steps, lr={lr}  ->  x = {x:.4f} (minimum at -1.5)")
plt.show()

## The learning rate is the whole story

Chapter 3 will show this again with a real model. It is cheaper to learn it here, on a parabola, where nothing takes two minutes to fail.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, lr in zip(axes, [0.02, 0.15, 0.95]):
    x, path = 4.0, [4.0]
    for _ in range(25):
        x = x - lr * df_exact(x)
        path.append(x)
    ax.plot(xs, f(xs), lw=1.2)
    ax.plot(path, [f(p) for p in path], "o-", ms=3.5, lw=.8, color="#c0392b")
    ax.set_title(f"lr = {lr}  ->  {x:.3f}")
    ax.set_ylim(-2, 40)
plt.tight_layout(); plt.show()

Too small and it never arrives. Too large and it oscillates or diverges. **There is no correct value in the abstract** — only one that suits the curvature of the surface you happen to be on.

## A gradient: the same idea in many dimensions

In [ ]:
import keras
from keras import ops

# Keras 3 exposes autodiff through the backend; here is the shape of it.
import tensorflow as tf

W = tf.Variable(tf.random.normal((2, 1)))
b = tf.Variable(tf.zeros((1,)))
X = tf.random.normal((64, 2))
y = X @ tf.constant([[2.0], [-3.0]]) + 0.5

with tf.GradientTape() as tape:
    pred = X @ W + b
    loss = tf.reduce_mean(tf.square(pred - y))

gW, gb = tape.gradient(loss, [W, b])
print("loss:", float(loss))
print("dL/dW:", gW.numpy().ravel())
print("dL/db:", gb.numpy())

`tape.gradient` is backpropagation. It applies the chain rule backwards through every operation recorded in the block above it — which is why the loss has to be **differentiable**, and why that constraint reappears in chapter 19 as a limitation rather than a detail.

## Full-batch, mini-batch, and stochastic

In [ ]:
def descend(batch_size, steps=120, lr=0.08, seed=0):
    rng = np.random.default_rng(seed)
    Xd = rng.normal(size=(512, 2))
    yd = Xd @ np.array([2.0, -3.0]) + 0.5 + rng.normal(scale=.3, size=512)
    w = np.zeros(2); bb = 0.0; hist = []
    for s in range(steps):
        idx = rng.choice(len(Xd), size=batch_size, replace=False)
        xb, yb = Xd[idx], yd[idx]
        err = xb @ w + bb - yb
        w -= lr * (2 * xb.T @ err / len(xb))
        bb -= lr * (2 * err.mean())
        hist.append(((Xd @ w + bb - yd) ** 2).mean())
    return hist

plt.figure(figsize=(7, 4))
for bs, label in [(1, "stochastic (1)"), (32, "mini-batch (32)"), (512, "full batch")]:
    plt.plot(descend(bs), lw=1.3, label=label)
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("full-dataset MSE")
plt.legend(); plt.title("Batch size trades noise against cost per step")
plt.show()

Full batch gives the smoothest curve and the most expensive step. Stochastic is noisy and cheap. **Mini-batch is neither, on purpose**, and it is what every model in this course uses.

---

## What to take away

- A derivative tells you which way to move; gradient descent does nothing else.
- The learning rate has no correct value in the abstract — too small stalls, too large diverges.
- Backpropagation is the chain rule applied backwards over recorded operations, which is why the whole chain must be differentiable.
- Batch size trades gradient noise against cost per step; mini-batch is the deliberate middle.